### [Beating Buy-and-Hold With a Simple Calendar-Based Strategy For 472% Returns](https://medium.com/coding-nexus/beating-buy-and-hold-with-a-simple-calendar-based-strategy-for-472-returns-3c7bf031bde1)

> How month-end institutional flows in U.S. Treasuries reveal a powerful lesson in systematic trading

In [1]:
!pip install --quiet vectorbt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.8/527.8 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.5/315.5 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 18.2 MB/s eta 0:00:00


In [2]:
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import vectorbt as vbt

warnings.filterwarnings("ignore")

# Download historical price data for TLT ETF from Yahoo Finance and extract the closing prices
tlt = (
    vbt.YFData.download("TLT", start="2004-01-01", end="2024-12-01")
    .get("Close")
    .to_frame()
)
close = tlt.Close

# Set up empty dataframes to hold trading signals for short and long positions
short_entries = pd.DataFrame.vbt.signals.empty_like(close)
short_exits = pd.DataFrame.vbt.signals.empty_like(close)
long_entries = pd.DataFrame.vbt.signals.empty_like(close)
long_exits = pd.DataFrame.vbt.signals.empty_like(close)

# Generate short entry signals on the first day of each new month
short_entry_mask = ~tlt.index.tz_convert(None).to_period("M").duplicated()
short_entries.iloc[short_entry_mask] = True

# Generate short exit signals five days after short entry
short_exit_mask = short_entries.shift(5).fillna(False)
short_exits.iloc[short_exit_mask] = True

# Generate long entry signals seven days before the end of each month
long_entry_mask = short_entries.shift(-7).fillna(False)
long_entries.iloc[long_entry_mask] = True

# Generate long exit signals one day before the end of each month
long_exit_mask = short_entries.shift(-1).fillna(False)
long_exits.iloc[long_exit_mask] = True

# Run the simulation and calculate performance statistics
pf = vbt.Portfolio.from_signals(
    close=close,
    entries=long_entries,
    exits=long_exits,
    short_entries=short_entries,
    short_exits=short_exits,
    freq="1d",
)

pf.stats()
pf.plot().show()